# 🧠 MiniLLM — notebook **Kaggle** (GPU T4 ×2)

Kaggle **ne peut pas monter Google Drive**. Ce notebook récupère donc tes données / checkpoints de trois façons, dans cet ordre :
1. **`/kaggle/input`** : un Dataset Kaggle ou la **sortie d'une version précédente de ce notebook** (le plus simple pour enchaîner les sessions) ;
2. **Google Drive via liens de partage** (`gdown`, transfert serveur à serveur : rien ne passe par ton téléphone) — à faire **une seule fois** ;
3. sinon, **reconstruction depuis internet** (uniquement si aucun checkpoint n'existe).

### Réglages Kaggle (panneau de droite)
* **Accelerator : GPU T4 ×2** · **Internet : On** (compte vérifié par téléphone) · **Persistence : Files only**.

### Enchaîner les sessions (tu ne perds rien)
1. Configure la cellule 1, puis **Save Version → Save & Run All (Commit)**. L'entraînement s'arrête proprement à `MAX_MINUTES` (avant la limite de 12 h) ; tout `/kaggle/working` est conservé.
2. Session suivante : **Add Input → Notebook Output Files → ce notebook**. Le code recopie tout seul le dernier checkpoint et les données, puis **reprend**.
3. Quand le pré-entraînement est fini : `PHASE = "sft"`, puis `PHASE = "chat"`.

### Phases (une par exécution, pour qu'un « Run All » soit toujours sûr)
`data` → préparer/récupérer les données · `pretrain` · `sft` · `chat` (évaluation, démo, RAG) · `all` (tout, si le temps le permet)

## 1 · Configuration

In [ ]:
import os, sys, json, time, shutil, subprocess

WORK = "/kaggle/working/MiniLLM"        # tout ce qui est ici est conservé par « Save Version »
CODE = "/kaggle/working/minillm_code"
os.makedirs(WORK, exist_ok=True)

# ══ Ce que fait cette exécution ═══════════════════════════════════════════
PHASE  = "pretrain"     # "data" | "pretrain" | "sft" | "chat" | "all"
BRANCH = "main"         # "main" = v2 (reprendre ton run Colab)   |   "v2.1" = nouvelle version (run neuf, voir README)
REPO   = "https://github.com/bono-p/minillm_v2.git"

# ══ Pré-entraînement ══════════════════════════════════════════════════════
MODEL_SIZE = "49M"
PT_ITERS   = 8_000       # v2 : reprise de l'it 4000 -> 8000 (≈ 2 h sur T4×2, 3,5 h sur 1 GPU) ; run neuf v2.1 : 10_000
SCHEDULE   = "cosine"    # "cosine" pour reprendre le run v2 ; "wsd" conseillé pour un run neuf en v2.1 (arrêtable à tout moment)
PT_LR      = 6e-4
PT_BATCH   = 16          # par GPU ; 8 si « out of memory »
PT_TOKENS_PER_STEP = 65_536
MAX_MINUTES = 640        # arrêt propre avant la limite de 12 h (setup + sauvegardes inclus)

# ══ Données (utilisé seulement si elles doivent être reconstruites : run neuf) ═══
WIKI_DOCS, WEB_DOCS, VOCAB_SIZE = 250_000, 300_000, 32_000

# ══ Fine-tuning ═══════════════════════════════════════════════════════════
SFT_EPOCHS = 3
SFT_ALPACA, SFT_OASST, SFT_PIAF = 30_000, 3_000, 4_000
PERSONA_FILE, PERSONA_REPEAT = "personnalite.jsonl", 20

# ══ Google Drive → Kaggle (une seule fois) ════════════════════════════════
# Dans l'app Drive : ⋮ sur le fichier → Partager → « Accès général : Toute personne disposant du lien » (Lecteur) → Copier le lien.
# Colle les liens ci-dessous (laisse "" pour ignorer). Tu peux retirer le partage ensuite. Le nom du checkpoint doit rester ckpt_XXXXXXX.pt.
DRIVE = {
    "data/tokenizer.json":                  "",
    "data/pretrain/meta.json":              "",
    "data/pretrain/val.bin":                "",
    "data/pretrain/train.bin":              "",          # ~670 Mo
    "checkpoints/pretrain/ckpt_0004000.pt": "",          # ~585 Mo : état complet pour REPRENDRE
    "checkpoints/pretrain/best.pt":         "",
    "checkpoints/pretrain/best.json":       "",
    "checkpoints/pretrain/log.jsonl":       "",
}

PT_DATA, PT_OUT = f"{WORK}/data/pretrain", f"{WORK}/checkpoints/pretrain"
SFT_DATA, SFT_OUT = f"{WORK}/data/sft", f"{WORK}/checkpoints/sft"

def phase_on(*names):
    return PHASE == "all" or PHASE in names
assert PHASE in ("data", "pretrain", "sft", "chat", "all"), "PHASE inconnue"
print(f"PHASE={PHASE} | BRANCH={BRANCH} | WORK={WORK}")

## 2 · Installation, code, GPU

In [ ]:
!pip -q install tokenizers datasets gdown
!rm -rf {CODE}
!git clone -q --depth 1 -b {BRANCH} {REPO} {CODE}
if BRANCH != "v2.1":   # les outils Kaggle (récupération données/checkpoints) vivent dans la branche v2.1
    !git -C {CODE} fetch -q --depth 1 origin v2.1 && git -C {CODE} show FETCH_HEAD:kaggle_utils.py > {CODE}/kaggle_utils.py
os.chdir(CODE); sys.path.insert(0, CODE)
os.environ.update(NCCL_P2P_DISABLE="1", NCCL_IB_DISABLE="1", TOKENIZERS_PARALLELISM="false")   # évite les blocages de torchrun sur Kaggle

import torch
N_GPU = torch.cuda.device_count()
print("GPU :", [torch.cuda.get_device_name(i) for i in range(N_GPU)] or "AUCUN")
if N_GPU == 0 and phase_on("pretrain", "sft"):
    raise RuntimeError("Aucun GPU : Settings → Accelerator → GPU T4 x2, puis relance.")
PT_ACCUM = max(1, PT_TOKENS_PER_STEP // (PT_BATCH * 512 * max(1, N_GPU)))
print(f"Batch effectif : {PT_BATCH} x {PT_ACCUM} x {max(1, N_GPU)} GPU x 512 = {PT_BATCH * PT_ACCUM * max(1, N_GPU) * 512:,} tokens/pas")

## 3 · Récupérer les données / le checkpoint (une seule fois)
Vérifie aussi que rien n'est tronqué. **Sécurité** : si un checkpoint existe mais pas ses données, le code refuse de reconstruire
(un tokenizer reconstruit aurait la même taille mais un autre contenu → modèle corrompu en silence).

In [ ]:
import kaggle_utils as ku
from checkpoint import list_checkpoints

if phase_on("data", "pretrain", "sft", "chat"):
    print("1) Entrées Kaggle (/kaggle/input)…")
    found = ku.restore_from_inputs("/kaggle/input", WORK)
    print("  ->", found or "rien de nouveau à copier")
    if any(v.strip() for v in DRIVE.values()):
        print("2) Google Drive (gdown)…")
        ku.download_drive(DRIVE, WORK)

    if os.path.exists(f"{PT_DATA}/meta.json"):
        meta = ku.verify_pretrain_data(PT_DATA)
        print(f"✓ données de pré-entraînement OK : {meta['n_train_tokens']:,} tokens d'entraînement")
    ck = list_checkpoints(PT_OUT)
    if ck:
        print(f"✓ checkpoint de reprise : it {ck[-1][0]} ->", ku.verify_checkpoint(ck[-1][1]))

In [ ]:
# Reconstruction complète (run NEUF uniquement)
need_data = not (os.path.exists(f"{PT_DATA}/meta.json") and os.path.exists(f"{WORK}/data/tokenizer.json"))
if phase_on("data", "pretrain") and need_data:
    if list_checkpoints(PT_OUT):
        raise RuntimeError("Un checkpoint existe mais pas les données/tokenizer qui vont avec : fournis-les (Drive ou /kaggle/input) "
                           "au lieu de les reconstruire, sinon le tokenizer ne correspondrait plus au modèle.")
    !python prepare_data.py all --out {WORK}/data --wiki_docs {WIKI_DOCS} --web_docs {WEB_DOCS} --vocab_size {VOCAB_SIZE}
ku.summarize(WORK)

## 4 · Pré-entraînement
`--resume auto` reprend le dernier checkpoint. À `MAX_MINUTES` : sauvegarde propre puis arrêt → **Save Version** et enchaîne.
Reprendre un run 1 GPU (Colab) sur 2 GPU (Kaggle) est sûr : le flux de données est identique (testé).

In [ ]:
if phase_on("pretrain"):
    args = (f"--mode pretrain --data_dir {PT_DATA} --out_dir {PT_OUT} --model_size {MODEL_SIZE} "
            f"--seq_len 512 --batch_size {PT_BATCH} --grad_accum {PT_ACCUM} --lr {PT_LR} --min_lr {PT_LR/10} "
            f"--max_iters {PT_ITERS} --warmup_iters 300 --schedule {SCHEDULE} --eval_every 500 --save_every 500 --eval_iters 50 "
            f"--max_minutes {MAX_MINUTES}")
    if N_GPU > 1:
        !torchrun --standalone --nproc_per_node={N_GPU} train.py {args}
    else:
        !python train.py {args}
    ku.prune_checkpoints(PT_OUT, keep=1)      # garde la sortie légère (le dernier checkpoint complet + best + final)
    ku.summarize(WORK)
    print("\n➡️  Maintenant : Save Version → Save & Run All. Prochaine session : Add Input → Notebook Output Files → ce notebook.")

In [ ]:
# Courbes (log.jsonl)
if os.path.exists(f"{PT_OUT}/log.jsonl") and phase_on("pretrain", "chat"):
    import matplotlib.pyplot as plt
    rows = [json.loads(l) for l in open(f"{PT_OUT}/log.jsonl")]
    tr = [(r["it"], r["loss"]) for r in rows if r["type"] == "train"]
    ev = [(r["it"], r["val_loss"], r["improved"]) for r in rows if r["type"] == "eval"]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(*zip(*tr), lw=1); ax[0].set_title("loss d'entraînement")
    ax[1].plot([e[0] for e in ev], [e[1] for e in ev], "o-", ms=3); ax[1].set_title("val_loss (val FIXE)"); plt.show()

## 5 · Fine-tuning « questions → réponses » (+ personnalité)

In [ ]:
if phase_on("sft"):
    if not os.path.exists(f"{SFT_DATA}/meta.json"):
        !python sft_data.py --tokenizer {WORK}/data/tokenizer.json --out {SFT_DATA} --alpaca {SFT_ALPACA} --oasst {SFT_OASST} --piaf {SFT_PIAF} --persona {PERSONA_FILE} --persona_repeat {PERSONA_REPEAT}
    !python train.py --mode sft --data_dir {SFT_DATA} --out_dir {SFT_OUT} --init_from {PT_OUT}/best.pt --epochs {SFT_EPOCHS}
    ku.prune_checkpoints(SFT_OUT, keep=1)
    ku.summarize(WORK)

## 6 · Évaluation, démo, mini-RAG, export

In [ ]:
if phase_on("chat"):
    !python evaluate.py qa      --ckpt {SFT_OUT}/best.pt --sft_dir {SFT_DATA} --n 300
    !python evaluate.py demo    --ckpt {SFT_OUT}/best.pt
    !python evaluate.py persona --ckpt {SFT_OUT}/best.pt || true
    !python rag.py --ckpt {SFT_OUT}/best.pt --kb knowledge --question "Quelle est la capitale du Cameroun ?" || true

In [ ]:
if phase_on("chat"):
    os.makedirs(f"{WORK}/export", exist_ok=True)
    for f in ("best.pt", "tokenizer.json"):
        shutil.copy(f"{SFT_OUT}/{f}", f"{WORK}/export/{f}")
    print(shutil.make_archive(f"{WORK}/minillm_export", "zip", f"{WORK}/export"))

## Dépannage Kaggle
* **`gdown` : « Cannot retrieve the public link »** → le fichier n'est pas partagé en « Toute personne disposant du lien », ou quota Drive atteint (réessaie plus tard).
* **`torchrun` bloque** → mets `N_GPU = 1` dans la cellule 2 (après le `import torch`) et relance.
* **Out of memory** → `PT_BATCH = 8`.
* **« Tokenizer incompatible »** → tes données ne correspondent pas au checkpoint : reprends **les mêmes** `tokenizer.json` / `train.bin` que ceux de l'entraînement.
* **La session a dépassé 12 h** → baisse `MAX_MINUTES` ; le dernier checkpoint (toutes les 500 it) est de toute façon conservé si tu as fait *Save Version*.